In [34]:
import pandas as pd

fire = pd.read_csv("fire.csv", parse_dates=['dt'])
fire.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 658788 entries, 0 to 658787
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   dt         658788 non-null  datetime64[ns]
 1   type_name  658788 non-null  object        
 2   type_id    658788 non-null  int64         
 3   lon        658788 non-null  float64       
 4   lat        658788 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 25.1+ MB


In [27]:
fire

,dt,type_name,type_id,lon,lat
0,2012-03-13,Природный пожар,4,131.5866,47.8662
1,2012-03-13,Природный пожар,4,131.5885,47.8809
2,2012-03-13,Лесной пожар,3,131.9871,48.4973
3,2012-03-13,Природный пожар,4,131.9031,43.6277
4,2012-03-13,Природный пожар,4,131.5706,47.8581
...,...,...,...,...,...
658783,2021-09-10,Лесной пожар,3,118.5451,64.7475
658784,2021-09-10,Лесной пожар,3,118.3046,64.7629
658785,2021-09-10,Лесной пожар,3,117.9681,65.7394
658786,2021-09-10,Лесной пожар,3,119.0462,64.7541


In [28]:
import psycopg2

connection = psycopg2.connect(user="postgres", password="admin", host="localhost", port=5432, database="tourism")
cursor = connection.cursor()

fire = fire.values.tolist()

sql_create_table_fire = """
CREATE TABLE public.fire(
    dt timestamp,
    type_name text,
    type_id integer,
    lon float,
    lat float
);
"""
cursor.execute(sql_create_table_fire)
connection.commit()

for i, row in enumerate(fire): 
    sql_insert_fire = """ 
    INSERT INTO public.fire(
        dt,
        type_name,
        type_id,
        lon,
        lat
    )
    VALUES (%s, %s, %s, %s, %s);
    """
    cursor.execute(sql_insert_fire, (row[0], row[1], row[2], row[3], row[4]))
    connection.commit()

connection.close()



In [ ]:
import psycopg2
import os

# Подключение к базе
connection = psycopg2.connect(user="postgres", password="admin", host="localhost", port=5432, database="tourism")
cursor = connection.cursor()

folder_path = 'foto'

sql_create_table_photo = """
CREATE TABLE IF NOT EXISTS public.photos(
    id SERIAL PRIMARY KEY,
    filename TEXT,
    photo_data BYTEA
);
"""
cursor.execute(sql_create_table_photo)
connection.commit()

files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff'))]

for i, filename in enumerate(files):
    file_path = os.path.join(folder_path, filename)

    with open(file_path, 'rb') as f:
        binary_data = f.read()
    
    sql_insert_photo = """ 
    INSERT INTO public.photos(
        filename,
        photo_data
    )
    VALUES (%s, %s);
    """

    cursor.execute(sql_insert_photo, (filename, binary_data))
    connection.commit()
    
    print(f"Загружен файл {i+1}: {filename}")

cursor.close()
connection.close()


Загружен файл 1: IMG_3101.jpg
Загружен файл 2: IMG_3103.jpg
Загружен файл 3: IMG_3106.jpg
Загружен файл 4: IMG_3107.jpg
Загружен файл 5: LC08_L1TP_173028_20171231_20200902_02_T1_refl.tif
